In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi

Sun Aug 23 16:26:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [3]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 102.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.5 MB/s eta 0:00:00


In [4]:
import transformers
import datasets
import peft
import accelerate
import trl
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

Transformers: 5.15.1
Datasets: 5.0.1
PEFT: 0.20.0
Accelerate: 1.14.0
TRL: 1.10.0
bitsandbytes: 0.50.1


In [13]:
!pip install -q "transformers==4.57.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.


In [35]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.5 MB/s eta 0:00:00:00:01


In [2]:
!pip install -q "bitsandbytes==0.50.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 47.3 MB/s eta 0:00:00:00:0100:01


In [36]:
import torchao
import peft
import trl

print("torchao:", torchao.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


torchao: 0.18.0
PEFT: 0.20.0
TRL: 1.10.0


In [6]:
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)

bitsandbytes: 0.50.1


In [5]:
import huggingface_hub
print(huggingface_hub.__version__)

1.11.0


In [7]:
from transformers.utils import is_bitsandbytes_available

print("bitsandbytes available:", is_bitsandbytes_available())

bitsandbytes available: False


In [1]:
import torch
import transformers
import datasets
import peft
import bitsandbytes

from transformers.utils import is_bitsandbytes_available

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

print("CUDA available:", torch.cuda.is_available())
print("Transformers sees bnb:", is_bitsandbytes_available())

ModuleNotFoundError: No module named 'bitsandbytes'

In [8]:
import bitsandbytes as bnb
import torch

print("bitsandbytes version:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

try:
    print("bnb CUDA setup:")
    print(bnb.cuda_specs())
except Exception as e:
    print("bnb error:", repr(e))

bitsandbytes version: 0.50.1
CUDA available: True
CUDA version: 12.8
bnb CUDA setup:
bnb error: TypeError("'module' object is not callable")


In [9]:
import bitsandbytes as bnb
import bitsandbytes.functional as F

print("bitsandbytes:", bnb.__version__)

try:
    print("CUDA version detected by bnb:", bnb.cuda_version_string())
except Exception as e:
    print("CUDA version check failed:", repr(e))

try:
    print("Available CUDA libs:")
    print(bnb.lib)
except Exception as e:
    print("Library check failed:", repr(e))

try:
    x = torch.randn(2, 2, device="cuda")
    print("CUDA tensor test:", x)
except Exception as e:
    print("CUDA tensor failed:", repr(e))

bitsandbytes: 0.50.1
CUDA version check failed: AttributeError("module 'bitsandbytes' has no attribute 'cuda_version_string'")
Available CUDA libs:
Library check failed: AttributeError("module 'bitsandbytes' has no attribute 'lib'")
CUDA tensor test: tensor([[ 1.4987, -0.9809],
        [-1.8080,  0.3656]], device='cuda:0')


In [10]:
import importlib.util
import transformers

print("Transformers:", transformers.__version__)

spec = importlib.util.find_spec("bitsandbytes")
print("bitsandbytes found:", spec is not None)
print("bitsandbytes path:", spec.origin if spec else None)

try:
    import bitsandbytes
    print("bitsandbytes import: OK")
except Exception as e:
    print("bitsandbytes import FAILED:", repr(e))

from transformers.utils import is_bitsandbytes_available

print("Transformers sees bitsandbytes:",
      is_bitsandbytes_available())

Transformers: 5.0.0
bitsandbytes found: True
bitsandbytes path: /usr/local/lib/python3.12/dist-packages/bitsandbytes/__init__.py
bitsandbytes import: OK
Transformers sees bitsandbytes: False


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face authentication successful")

Hugging Face authentication successful


In [38]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(type(tokenizer))
print("Pad token:", tokenizer.pad_token)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("Model loaded successfully")
print("Device:", model.device)

<class 'transformers.models.gemma.tokenization_gemma.GemmaTokenizer'>
Pad token: <pad>


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded successfully
Device: cuda:0


In [39]:
messages = [
    {
        "role": "user",
        "content": "Explain what Kubernetes HPA is in simple terms."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

# Move every input tensor to the model's device
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Explain what Kubernetes HPA is in simple terms.
model
Okay, let's break down Kubernetes Horizontal Pod Autoscaler (HPA) in simple terms.

**Imagine you're running a really popular website.**  Lots of people are trying to access it at the same time – it's a lot of traffic. 

**Without HPA,** your website might be struggling – some pages might be slow, some might crash, and you’ll lose users.

**That's where HPA comes in.**  It’s


In [3]:
from datasets import load_dataset

iris_v1 = load_dataset(
    "json",
    data_files="/kaggle/input/datasets/sandeepnalla6125/training-data/iris_v1.jsonl"
)

print(iris_v1)
print(iris_v1["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_text', 'output_text'],
        num_rows: 150
    })
})
{'input_text': 'sepal_length: 5.1, sepal_width: 3.5, petal_length: 1.4, petal_width: 0.2', 'output_text': 'setosa'}


In [16]:
from datasets import load_dataset

iris_v2 = load_dataset(
    "json",
    data_files="/kaggle/input/datasets/sandeepnalla6125/training-data/iris_v2.jsonl"
)

print(iris_v2)
print(iris_v2["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input_text', 'output_text'],
        num_rows: 150
    })
})
{'input_text': 'A flower specimen has a sepal length of 5.1 cm, sepal width of 3.5 cm, petal length of 1.4 cm, and petal width of 0.2 cm. Identify the iris species.', 'output_text': 'This is Iris setosa.'}


In [25]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [26]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [41]:
def format_example(example):
    return {
        "text": (
            "Classify the following Iris flower.\n\n"
            f"{example['input_text']}\n\n"
            "Species: "
            f"{example['output_text']}"
        )
    }

train_dataset = iris_v1["train"].map(format_example)

print(train_dataset[0]["text"])

Classify the following Iris flower.

sepal_length: 5.6, sepal_width: 2.7, petal_length: 4.2, petal_width: 1.3

Species: versicolor


In [28]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/kaggle/working/gemma-iris-v1",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

In [43]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


AttributeError: 'functools.partial' object has no attribute '__func__'

In [12]:
# ============================================================
# Gemma 3 1B - QLoRA Fine-tuning on Iris Dataset
# ============================================================

import torch

from datasets import DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig, get_peft_model


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-3-1b-it"

OUTPUT_DIR = "/kaggle/working/gemma-iris"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded")


# ------------------------------------------------------------
# 3. 4-bit quantization configuration
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


# ------------------------------------------------------------
# 4. Load Gemma model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False

print("Model loaded")


# ------------------------------------------------------------
# 5. Prepare training text
# ------------------------------------------------------------

def format_example(example):
    return {
        "text": (
            "Classify the following Iris flower.\n\n"
            f"{example['input_text']}\n\n"
            f"Species: {example['output_text']}"
        )
    }


train_dataset = iris_v1["train"].map(
    format_example
)

print("\nExample training text:")
print(train_dataset[0]["text"])


# ------------------------------------------------------------
# 6. Tokenize dataset
# ------------------------------------------------------------

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
    )


tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

print("\nTokenized example:")
print(tokenized_dataset[0])


# ------------------------------------------------------------
# 7. Configure LoRA
# ------------------------------------------------------------

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


# ------------------------------------------------------------
# 8. Apply LoRA ONCE
# ------------------------------------------------------------

model = get_peft_model(
    model,
    peft_config,
)

print("\nTrainable parameters:")
model.print_trainable_parameters()


# ------------------------------------------------------------
# 9. Data collator
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# ------------------------------------------------------------
# 10. Training configuration
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    fp16=True,

    report_to="none",

    optim="paged_adamw_8bit",
)


# ------------------------------------------------------------
# 11. Create standard Hugging Face Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)


# ------------------------------------------------------------
# 12. Verify trainer/batch before training
# ------------------------------------------------------------

print("\nTrainer created successfully")

batch = next(iter(trainer.get_train_dataloader()))

print("\nBatch:")
for key, value in batch.items():
    print(
        key,
        tuple(value.shape) if hasattr(value, "shape") else type(value)
    )


# ------------------------------------------------------------
# 13. Start fine-tuning
# ------------------------------------------------------------

print("\nStarting fine-tuning...")

trainer.train()

print("\nFine-tuning completed!")


# ------------------------------------------------------------
# 14. Save LoRA adapter
# ------------------------------------------------------------

ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("\nAdapter saved to:")
print(ADAPTER_DIR)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Tokenizer loaded


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`